# Notebook 01 — Carga y preparación de datos del IBWC

**Reto:** Falcon Challenge — Resilient Release Scheduling  
**Equipo 04 — Hackathon OQI 2026**

Este notebook carga los cuatro archivos CSV del IBWC, los limpia, los resamplea a resolución semanal y exporta las series temporales listas para usar en los notebooks de optimización.

## Datos utilizados
| Archivo | Columna clave | Uso |
|---|---|---|
| `Discharge.Best Available@08461300` | `Discharge_m3s` (m³/s) | $R^{obs}(t)$ — liberación histórica |
| `Total Storage.Web-Daily-tcm@08461200` | `Storage_m3` (m³) | $S(t)$ y $\Delta S^{obs}(t)$ |
| `Reservoir Elevation.Web-Daily-m@08461200` | `Elevation_m` (m) | Referencia para $S_{max}$ |
| `Lake Area.Best Available@08461200` | `LakeArea_m2` (m²) | Información complementaria |

## Salidas de este notebook
- `R_obs_semanal` — volumen de liberación por semana (m³/semana)
- `delta_S_obs_semanal` — cambio en almacenamiento por semana (m³)
- `S0`, `Smax`, `Smin` — condición inicial y límites físicos del embalse

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ── AJUSTA ESTA RUTA a tu entorno local ─────────────
RUTA_DATOS = r'C:\Users\Emdro\OneDrive\Escritorio\Equipo_04\FalconChallenge\data'

ARCHIVOS = {
    'discharge': 'DataSetExport-Discharge.Best Available@08461300-Instantaneous-m^3 s-20260629185451.csv',
    'lake_area': 'DataSetExport-Lake Area.Best Available@08461200-Instantaneous-m^2-20260629185344.csv',
    'elevation': 'DataSetExport-Reservoir Elevation.Web-Daily-m@08461200-Instantaneous-m-20260629185508.csv',
    'storage':   'DataSetExport-Total Storage.Web-Daily-tcm@08461200-Instantaneous-m^3-20260629185416.csv',
}

print('Archivos disponibles en la carpeta:')
for f in os.listdir(RUTA_DATOS):
    print(' ', f)

In [ ]:
def cargar_ibwc_robusto(nombre_archivo, nombre_valor):
    """
    Carga un CSV exportado del portal IBWC/Aquarius.
    Estrategia robusta: parsea fila a fila como fecha y descarta
    cualquier línea de metadata o disclaimer que no sea parseable.
    Funciona independientemente del número de filas de encabezado.
    """
    ruta_completa = os.path.join(RUTA_DATOS, nombre_archivo)
    df_raw = pd.read_csv(
        ruta_completa,
        header=None,
        names=['col1', 'col2'],
        skip_blank_lines=True,
        on_bad_lines='skip'
    )
    fechas = pd.to_datetime(df_raw['col1'], errors='coerce', format='mixed')
    mask   = fechas.notna()
    df     = df_raw[mask].copy()
    df['Timestamp']   = fechas[mask]
    df[nombre_valor]  = pd.to_numeric(df['col2'], errors='coerce')
    df = (df[['Timestamp', nombre_valor]]
            .dropna()
            .set_index('Timestamp')
            .sort_index())
    print(f'  {nombre_archivo[:50]}...')
    print(f'  → {len(df)} filas | {df.index.min()} a {df.index.max()}')
    return df

print('Cargando datasets...')
df_discharge = cargar_ibwc_robusto(ARCHIVOS['discharge'], 'Discharge_m3s')
df_lake_area = cargar_ibwc_robusto(ARCHIVOS['lake_area'], 'LakeArea_m2')
df_elevation = cargar_ibwc_robusto(ARCHIVOS['elevation'], 'Elevation_m')
df_storage   = cargar_ibwc_robusto(ARCHIVOS['storage'],   'Storage_m3')

In [ ]:
# ── Limpieza: eliminar duplicados y NaNs ─────────────
def limpiar(df, col):
    n_antes = len(df)
    df = df[~df.index.duplicated(keep='first')].dropna(subset=[col])
    print(f'{col}: {n_antes} → {len(df)} filas tras limpieza')
    return df

df_discharge = limpiar(df_discharge, 'Discharge_m3s')
df_storage   = limpiar(df_storage,   'Storage_m3')
df_elevation = limpiar(df_elevation, 'Elevation_m')

In [ ]:
# ── Conversión de unidades y resampleo a semanal ─────
#
# Discharge viene en m³/s (caudal instantáneo cada 15 min).
# Necesitamos volumen total liberado por semana (m³/semana).
# Pasos: m³/s promedio diario → m³/día (× 86400 s) → m³/semana (suma 7 días)

discharge_diario    = df_discharge['Discharge_m3s'].resample('D').mean()
discharge_vol_diario = discharge_diario * 86400          # m³/día
R_obs_semanal       = discharge_vol_diario.resample('W').sum()   # m³/semana

# Storage viene en m³, medición diaria.
# Resampleamos a la media semanal y calculamos el cambio semana a semana.
storage_semanal     = df_storage['Storage_m3'].resample('W').mean()
delta_S_obs_semanal = storage_semanal.diff().dropna()

print('R_obs_semanal (primeras 5):')
print(R_obs_semanal.head())
print('\ndelta_S_obs_semanal (primeras 5):')
print(delta_S_obs_semanal.head())

In [ ]:
# ── Alinear fechas comunes entre Discharge y Storage ─
fechas_comunes = R_obs_semanal.index.intersection(delta_S_obs_semanal.index)

R_obs_final        = R_obs_semanal.loc[fechas_comunes].values
delta_S_obs_final  = delta_S_obs_semanal.loc[fechas_comunes].values

# Parámetros del embalse
S0   = storage_semanal.loc[fechas_comunes[0]]   # almacenamiento inicial
Smax = storage_semanal.max()                     # máximo observado en el dataset
Smin = 0.25 * Smax                              # 25% del máximo = nivel crítico

print(f'Semanas disponibles : {len(fechas_comunes)}')
print(f'Desde               : {fechas_comunes.min()}')
print(f'Hasta               : {fechas_comunes.max()}')
print(f'\nS0   = {S0:>20,.0f} m³')
print(f'Smax = {Smax:>20,.0f} m³')
print(f'Smin = {Smin:>20,.0f} m³')
print(f'\nNOTA: Smax se toma como el máximo histórico observado en la ventana de datos.')
print(f'La capacidad de conservación oficial del IBWC es ~2,666,203 acre-pies ≈ 3.29e9 m³.')
print(f'Diferencia = {abs(3.29e9 - Smax)/3.29e9*100:.1f}% respecto al valor oficial.')

In [ ]:
# ── Verificar que el problema es 'interesante' ───────
# Si el almacenamiento histórico nunca cae bajo Smin, Ccrit = 0 todo el tiempo
# y el problema se vuelve trivialmente resuelto por u=0.
S_check = S0
n_criticos = 0
for t in range(len(delta_S_obs_final)):
    S_check = np.clip(S_check + delta_S_obs_final[t], 0, Smax)
    if S_check < Smin:
        n_criticos += 1
print(f'Semanas donde S(t) < Smin (sin optimización): {n_criticos}')
if n_criticos == 0:
    print('NOTA: En la ventana actual el embalse no entra en zona crítica.')
    print('Ccrit = 0 para el histórico. El optimizador actuará principalmente sobre Cdev y Csmooth.')
    print('Considera usar una ventana histórica con sequía (e.g., 2020-2022) para resultados más ilustrativos.')

In [ ]:
# ── Visualización exploratoria ───────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
semanas = np.arange(len(fechas_comunes))

axes[0].plot(semanas, R_obs_final / 1e6, color='steelblue', lw=2)
axes[0].set_title('Liberación histórica semanal R_obs(t)', fontsize=12)
axes[0].set_ylabel('Volumen (millones m³/semana)')
axes[0].grid(alpha=0.3)

S_traj = np.zeros(len(delta_S_obs_final) + 1)
S_traj[0] = S0
for t in range(len(delta_S_obs_final)):
    S_traj[t+1] = np.clip(S_traj[t] + delta_S_obs_final[t], 0, Smax)

axes[1].plot(np.arange(len(S_traj)), S_traj / 1e9, color='royalblue', lw=2, label='S(t) observado')
axes[1].axhline(Smin / 1e9, color='red', ls=':', lw=2, label=f'Smin = {Smin/1e9:.3f} Gm³')
axes[1].axhline(Smax / 1e9, color='green', ls='--', lw=1.5, label=f'Smax = {Smax/1e9:.3f} Gm³')
axes[1].fill_between(np.arange(len(S_traj)), 0, Smin / 1e9, alpha=0.08, color='red', label='Zona crítica')
axes[1].set_title('Trayectoria de almacenamiento (sin optimización)', fontsize=12)
axes[1].set_ylabel('Almacenamiento (Gm³)')
axes[1].set_xlabel('Semana')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('01_exploracion_datos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfica guardada: 01_exploracion_datos.png')

In [ ]:
# ── Exportar datos procesados para los otros notebooks ─
np.save('R_obs_semanal.npy', R_obs_final)
np.save('delta_S_obs_semanal.npy', delta_S_obs_final)
np.save('params_embalse.npy', np.array([S0, Smax, Smin]))

print('Archivos exportados:')
print('  R_obs_semanal.npy     — liberaciones históricas semanales (m³/semana)')
print('  delta_S_obs_semanal.npy — cambios de almacenamiento semanales (m³)')
print('  params_embalse.npy    — [S0, Smax, Smin]')